In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
import warnings
warnings.filterwarnings("ignore")

from ultralytics import YOLO

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
import ultralytics
print(f"Versao do Yolo: {ultralytics.__version__}")

yolo_custom = None


def detectar_yolo_custom(image_path, model, labels=[0,1], limiar_confianca=0.4):
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    inicio = time.time()
    resultados = model(
        img_rgb,
        conf=limiar_confianca,
        classes=labels,
        verbose=False
    )
    inferencia_ms = (time.time() - inicio) * 1000  # em ms
    retangulos = resultados[0].boxes
    deteccoes = []
    if retangulos is not None:
        for retang in retangulos:
            x1, y1, x2, y2 = retang.xyxy[0].cpu().numpy().astype(int)
            conf = float(retang.conf[0])
            deteccoes.append({"retang": (x1, y1, x2, y2), "conf": conf, 'cls': int(retang.cls.cpu().item())})
    
    return {
        "img": img_rgb,
        "deteccoes": deteccoes,
        "inferencia_ms": inferencia_ms,
        "num_deteccoes": len(deteccoes)
    }


def visualizar_deteccoes_yolo(result, title="YOLOv8 customizado"):
    fig, ax = plt.subplots(1, 1, figsize=(9, 6))
    ax.imshow(result["img"])
    cores = plt.cm.Set1(np.linspace(0, 1, max(len(result["deteccoes"]), 1)))
    for i, det in enumerate(result["deteccoes"]):
        x1, y1, x2, y2 = det["retang"]
        conf = det["conf"]
        label_name = yolo_custom.names[det["cls"]]
        color = cores[i % len(cores)]
        # retangulo deteccao
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2.5, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        # rótulo com confiança
        ax.text(
            x1, y1 - 6, f"{label_name.title()} {conf:.2f}",
            color="white", fontsize=9, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.8)
        )
    
    ax.set_title(
        f"{title}\n"
        f"{result['num_deteccoes']} deteccoes | "
        f"Inferência: {result['inferencia_ms']:.1f}ms",
        fontsize=10, pad=12
    )
    ax.axis("off")
    plt.tight_layout()
    plt.savefig("output/yolo_deteccoes.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Salvo em output/yolo_deteccoes.png")

In [ ]:
# path do melhor modelo do treino com o dataset 1
# best_model_path = 'models/best1.pt'
# path do melhor modelo do treino com o dataset 2
best_model_path = 'models/best2.pt'

yolo_custom = YOLO(best_model_path)

print(60*'-')
print(f"Classes disponíveis:")
for id, name in yolo_custom.names.items():
    print(f"   ID {id}: {name}")

print(60*'-')
print(f"   Total de classes: {len(yolo_custom.names)}")

img_teste = "images/t10.jpg"
result = detectar_yolo_custom(img_teste, yolo_custom, labels=[0,1], limiar_confianca=0.4)
print(f"Resultado:")
print(f"   deteccoes: {result['num_deteccoes']}")
print(f"   Tempo de inferência: {result['inferencia_ms']:.1f}ms")
for i, d in enumerate(result["deteccoes"]):
    print(f"   [{i+1}] retang={d['retang']} | confiança={d['conf']:.3f}")
visualizar_deteccoes_yolo(result)